In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time
import UEG_response as ur

In [ ]:
# Conditions
rs = 3.23
theta = 1.0

# Units
hbar = 1.0
aB = 1.0
m = 1.0
e = 1.0

# Normalisation
qF = (9*np.pi/4)**(1/3) / (rs*aB)
EF = hbar**2 * qF**2 / (2*m)
beta = 1/(theta*EF)
n = 3/(4*np.pi*rs**3)

# Tolerances
reltol = 1e-16
abstol = 1e-8
eta_log  = 1e-6
eta_sqrt = 1e-6
eta_pol = 1e-4
points_n = 5
tol_upper = 1e-8
dx = 1e-4
lower = 1e-6
limit = 50



In [ ]:
# Test diagonal case.

ks = np.array([1.0, 1.5, 2.0]) * qF
omega = np.linspace(-14.0, 14.0, 201) * EF/hbar

fig, axs = plt.subplots(2, 1, sharex=True)
for k in ks:
    chi2_0_diag = ur.ideal_diagonal_quadratic_response(omega, k, m, hbar, n, beta,
                                                        ms=2, reltol=reltol, abstol=abstol, limit=limit,
                                                        eta_log=eta_log, tol_upper=tol_upper,
                                                        points_n=points_n, force_output=True)

    norm = n*beta**2
    axs[0].plot(omega*hbar/EF, np.real(chi2_0_diag)/norm, linewidth=2.0, label=f"$k = %.1f q_F$"%(k/qF))
    axs[1].plot(omega*hbar/EF, np.imag(chi2_0_diag)/norm, linewidth=2.0)

    chi2_0= ur.ideal_quadratic_response(omega, k, omega, k, 1.0,
                                        m, hbar, n, beta, ms=2,
                                        reltol=reltol, abstol=abstol, limit=limit, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper, lower=lower,
                                        dx=dx, points_n=points_n, force_output=True)

    axs[0].plot(omega*hbar/EF, np.real(chi2_0)/norm, '--k')
    axs[1].plot(omega*hbar/EF, np.imag(chi2_0)/norm, '--k')
    


axs[1].set_xlabel(r"$\omega$ [$E_F/\hbar$]")
axs[0].set_ylabel(r"Re$\chi_0^{(2)}(\vec{k}, \omega, \vec{k}, \omega)$ [$n \beta^2$]")
axs[1].set_ylabel(r"Im$\chi_0^{(2)}(\vec{k}, \omega, \vec{k}, \omega)$ [$n \beta^2$]")

axs[0].legend()


In [ ]:
# Test the Kramers–Kronig relations
# Settings for numerical integration
omega_max = 20 * EF/hbar
num_omega = 300

# Target conditions
np.random.seed(42)
k1 = np.random.uniform(0.0, 4.0, 1)[0] * qF
omega1 = np.random.uniform(-1.0, 1.0, 1)[0] * EF/hbar
k2 = np.random.uniform(0.0, 4.0, 1)[0] * qF
omega2 = np.random.uniform(-1.0, 1.0, 1)[0] * EF/hbar
csTheta = np.random.uniform(-1.0, 1.0, 1)[0]

print(f"Settings: ")
print(f"k1 = %g"%(k1))
print(f"omega1 = %g"%(omega1))
print(f"k2 = %g"%(k2))
print(f"omega2 = %g"%(omega2))
print(f"csTheta = %g"%(csTheta))

chi2_0 = ur.ideal_quadratic_response(omega1, k1, omega2, k2, csTheta,
                                        m, hbar, n, beta, ms=2,
                                        reltol=reltol, abstol=abstol, limit=limit, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper, lower=lower,
                                        dx=dx, points_n=points_n, force_output=True)[0]

# Kramers–Kronig in first argument
omega_p = omega1 + np.linspace(-omega_max, omega_max, num_omega)
chi2_0_p = ur.ideal_quadratic_response(omega_p, k1, omega2, k2, csTheta,
                                        m, hbar, n, beta, ms=2,
                                        reltol=reltol, abstol=abstol, limit=limit, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper, lower=lower,
                                        dx=dx, points_n=points_n, force_output=True)
chi2_0_KK = (-1j) * np.trapezoid(chi2_0_p/(omega_p - omega1), x=(omega_p-omega1)) / np.pi
rel_diff = (chi2_0 - chi2_0_KK) / (np.abs(chi2_0) + np.abs(chi2_0_KK))

print(f"\nFirst argument:")
print(f"Direct:         %g + %g j"%(np.real(chi2_0),np.imag(chi2_0)))
print(f"Kramers-Kronig: %g + %g j"%(np.real(chi2_0_KK),np.imag(chi2_0_KK)))
print(f"diff:           %g + %g j"%(np.real(chi2_0-chi2_0_KK),np.imag(chi2_0-chi2_0_KK)))
print(f"Rel. diff:      %g + %g j"%(np.real(rel_diff),np.imag(rel_diff)))

# Kramers–Kronig in second argument
omega_p = omega2 + np.linspace(-omega_max, omega_max, num_omega)
chi2_0_p = ur.ideal_quadratic_response(omega1, k1, omega_p, k2, csTheta,
                                        m, hbar, n, beta, ms=2,
                                        reltol=reltol, abstol=abstol, limit=limit, eta_pol=eta_pol, eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper, lower=lower,
                                        dx=dx, points_n=points_n, force_output=True)
chi2_0_KK = (-1j) * np.trapezoid(chi2_0_p/(omega_p - omega2), x=(omega_p-omega2)) / np.pi
rel_diff = (chi2_0 - chi2_0_KK) / (np.abs(chi2_0) + np.abs(chi2_0_KK))

print(f"\nSecond argument:")
print(f"Direct:         %g + %g j"%(np.real(chi2_0),np.imag(chi2_0)))
print(f"Kramers-Kronig: %g + %g j"%(np.real(chi2_0_KK),np.imag(chi2_0_KK)))
print(f"diff:           %g + %g j"%(np.real(chi2_0-chi2_0_KK),np.imag(chi2_0-chi2_0_KK)))
print(f"Rel. diff:      %g + %g j"%(np.real(rel_diff),np.imag(rel_diff)))


In [ ]:
# Compariosn between the different integration methods.
num_events = 10_000

# Raw inputs
np.random.seed(42)
omega1 = np.random.uniform(-10.0, 10.0, num_events)*EF/hbar
omega2 = np.random.uniform(-10.0, 10.0, num_events)*EF/hbar

k1 = np.random.uniform(0.001, 10.0, num_events)*qF
k2 = np.random.uniform(0.001, 10.0, num_events)*qF

csTheta = np.random.uniform(-1.0, 1.0, num_events)

# Time direct implementation
start = time.time()
chi2_0 = ur.ideal_quadratic_response(omega1, k1, omega2, k2, csTheta, 
                                 m, hbar, n, beta, ms=2, method='direct', use_parallel=False,
                                 reltol=reltol, abstol=abstol, limit=limit, eta_pol=eta_pol, 
                                 eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                 dx=dx, points_n=points_n, force_output=True)
end = time.time()
duration_old = end - start

# Time Maldague implementation
start = time.time()
chi2_0_M = ur.ideal_quadratic_response(omega1, k1, omega2, k2, csTheta,
                                 m, hbar, n, beta, method='maldague', ms=2, 
                                 reltol=reltol, abstol=abstol, limit=limit, eta_pol=eta_pol, 
                                 eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                 dx=dx, points_n=points_n, force_output=True)
end = time.time()
duration_new = end - start

# Plot error
rel_err = np.abs(chi2_0 - chi2_0_M) / (np.abs(chi2_0) + np.abs(chi2_0_M))

plt.hist(np.log10(rel_err))

plt.yscale('log')

plt.xlabel(r"$\log_{10}(|\chi^{D} - \chi^{M}|/(|\chi^{D}| + |\chi^{M}|))$")
plt.ylabel(r"Counts")


print(f"Theta = %s"%(str(theta)))
print(f"Duration old: %g ms / event"%(1e3*duration_old/num_events))
print(f"Duration new: %g ms / event"%(1e3*duration_new/num_events))
print(f"Ratio (new/old): %f"%(duration_new/duration_old))


In [ ]:
# Further comparisons between the implementations.

# Setup of some parameters
thetas = [20.0, 2.0, 1.0, 0.5, 0.1, 0.01]
colors = ['c', 'y', 'm', 'r', 'g', 'b']

# Inputs
z1 = 0.0
z2 = 0.1

y1 = 1.0
y2 = 1.5
csTheta = np.linspace(-0.999, 0.999, 100)

fig, axs = plt.subplots(1, 2, sharex=True, figsize=(2*6.4, 4.8))

# Classical limit
# Normalisation
qF = (9*np.pi/4)**(1/3) / (rs*aB)
EF = hbar**2 * qF**2 / (2*m)
beta = 1/(thetas[0]*EF)
n = 3/(4*np.pi*rs**3)
beta_eff = beta / np.sqrt(1 + (1/thetas[0])**2 )

# Input in physical units
omega1 = z1/(beta_eff * hbar)
omega2 = z2/(beta_eff * hbar)

k1 = y1*qF
k2 = y2*qF
classical_chi2_0 = ur.classical_ideal_quadratic_response(k1, omega1, k2, omega2, csTheta, n, beta, m, dc=dx, eta_pol=eta_pol, reltol=reltol, abstol=abstol)

norm = n*beta_eff**2
axs[0].plot(csTheta, np.real(classical_chi2_0)/norm, '-.', linewidth=2.0, color=colors[0], label=r"Classical: $\Theta = %.2f$"%(thetas[0]))
axs[1].plot(csTheta, np.imag(classical_chi2_0)/norm, '-.', linewidth=2.0, color=colors[0], label=r"Classical: $\Theta = %.2f$"%(thetas[0]))


# General temperatures
for i, _theta in enumerate(thetas):
    # Normalisation
    qF = (9*np.pi/4)**(1/3) / (rs*aB)
    EF = hbar**2 * qF**2 / (2*m)
    beta = 1/(_theta*EF)
    n = 3/(4*np.pi*rs**3)
    beta_eff = beta / np.sqrt(1 + (1/_theta)**2 )

    # Input in physical units
    omega1 = z1/(beta_eff*hbar)
    omega2 = z2/(beta_eff*hbar)

    k1 = y1*qF
    k2 = y2*qF

    chi2_0 = ur.ideal_quadratic_response(omega1, k1, omega2, k2, csTheta, 
                                        m, hbar, n, beta, ms=2, method='direct', use_parallel=True,
                                        reltol=reltol, abstol=abstol, eta_pol=eta_pol, 
                                        eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                        dx=dx, points_n=points_n, force_output=True)

    chi2_0_par = ur.ideal_quadratic_response(omega1, k1, omega2, k2, np.array([-1.0, 1.0]), 
                                            m, hbar, n, beta, ms=2, method='direct', use_parallel=True, 
                                            reltol=reltol, abstol=abstol, eta_pol=eta_pol, 
                                            eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                            dx=dx, points_n=points_n)

    chi2_0_M = ur.ideal_quadratic_response(omega1, k1, omega2, k2, csTheta, 
                                            m, hbar, n, beta, ms=2, method='maldague',
                                            reltol=reltol, abstol=abstol, eta_pol=eta_pol, 
                                            eta_sqrt=eta_sqrt, eta_log=eta_log, tol_upper=tol_upper,
                                            dx=dx, points_n=points_n, force_output=True)
    
    norm = n*beta_eff**2
    axs[0].plot(csTheta, np.real(chi2_0)/norm, '-', linewidth=2.0, color=colors[i], alpha=0.7, label=r"$\Theta = %.2f$"%(_theta))
    axs[0].plot(csTheta, np.real(chi2_0_M)/norm, ':', color='k')
    axs[0].plot(np.array([-1.0, 1.0]), np.real(chi2_0_par)/norm, 'o', color=colors[i])

    axs[1].plot(csTheta, np.imag(chi2_0)/norm, '-', linewidth=2.0, color=colors[i], alpha=0.7, label=r"$\Theta = %.2f$"%(_theta))
    axs[1].plot(csTheta, np.imag(chi2_0_M)/norm, ':', color='k')
    axs[1].plot(np.array([-1.0, 1.0]), np.imag(chi2_0_par)/norm, 'o', color=colors[i])

# Ground state results.
# Normalisation
qF = (9*np.pi/4)**(1/3) / (rs*aB)
EF = hbar**2 * qF**2 / (2*m)
n = 3/(4*np.pi*rs**3)
beta_eff = 1 / EF

# Input in physical units
omega1 = z1/(beta_eff*hbar)
omega2 = z2/(beta_eff*hbar)

k1 = y1*qF
k2 = y2*qF

# Increase resolution for ground state
csTheta = np.linspace(csTheta[0], csTheta[-1], 500)

ground_state_chi2_0 = ur.ground_state_ideal_quadratic_response(omega1, k1, omega2, k2, csTheta, m, hbar, n, ms=2)

norm = n*beta_eff**2
axs[0].plot(csTheta, np.real(ground_state_chi2_0)/norm, '--', linewidth=2.0, color='k', label=r"$T = 0$")
axs[1].plot(csTheta, np.imag(ground_state_chi2_0)/norm, '--', linewidth=2.0, color='k', label=r"$T = 0$")

axs[0].set_xlabel(r'$\cos\theta$')
axs[1].set_xlabel(r'$\cos\theta$')

axs[0].set_ylabel(r'Re$\{\chi^{(2)}_0(\vec{k}_1, \omega_1, \vec{k}_2, \omega_2)\}$ [$n\beta_{eff}$]')
axs[1].set_ylabel(r'Im$\{\chi^{(2)}_0(\vec{k}_1, \omega_1, \vec{k}_2, \omega_2)\}$ [$n\beta_{eff}$]')

axs[1].legend(ncol=2)


# plt.savefig(f"figures/implementation_comparison.jpg", dpi=400, bbox_inches='tight')
